# Диффузионная модель MNIST

Ноутбук самостоятельный: загружать дополнительные `.py` файлы не нужно.
Внутри находится та же DhariwalUNet и те же формулы EDM, что в нашем скрипте.
Интернет нужен для библиотек (если отсутствуют) и скачивания MNIST.

## Как запустить

1. Откройте https://colab.research.google.com/ и выберите **Файл → Загрузить блокнот**.
2. Загрузите `MNIST_Diffusion_Colab.ipynb`.
3. Выберите **Среда выполнения → Сменить среду выполнения → GPU**
   (`Runtime → Change runtime type → GPU`). Доступность ускорителя зависит от Colab.
4. Запускайте ячейки сверху вниз кнопкой ▶ или сочетанием **Shift+Enter**.
5. В разделе настроек для первого знакомства можно поставить `EPOCHS = 1`.
   Для основного обучения оставьте `6`. Одна эпоха не гарантирует хороших цифр.
6. После обучения посмотрите график и примеры, затем скачайте результаты в конце.

**Полное обучение запускается ячейкой «Обучение».** Предыдущие ячейки готовят
данные, определения функций и необученную модель.

## 1. Подготовка библиотек

Используем готовые PyTorch и torchvision среды Colab. Ячейка устанавливает только
отсутствующие библиотеки и не заменяет установленную CUDA-сборку CPU-сборкой.
Если среда имеет несовместимую пару torch/torchvision, импорт покажет ошибку.

In [ ]:
import importlib.util
import subprocess
import sys

requirements = {
    'torch': 'torch>=2.6', 'torchvision': 'torchvision>=0.21',
    'numpy': 'numpy>=1.26', 'PIL': 'Pillow>=10',
    'matplotlib': 'matplotlib>=3.8', 'tqdm': 'tqdm>=4.66',
}
missing = [requirement for module, requirement in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Библиотеки подготовлены. Python:', sys.version.split()[0])

## 2. Загрузить U-Net

Загружаем готовую архитектуру из NVIDIA EDM. 

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from hashlib import sha256

revision = '008a4e5316c8e3bfe61a62f874bddba254295afb'
base_url = f'https://raw.githubusercontent.com/NVlabs/edm/{revision}/'
vendor_dir = Path('vendor')
vendor_dir.mkdir(exist_ok=True)
downloads = [
    ('training/networks.py', 'edm_networks.py', '5db27dcd96674b95c72d5e6491b879cdc35e24039ada3411b4b46a28ed1fe284'),
    ('LICENSE.txt', 'EDM_LICENSE.txt', 'b7249043cca5e5c88f99730ddc40490a3284780dfe423b19f2c75bd5e2a8f03e'),
]
for remote, local, expected_hash in downloads:
    content = urlopen(base_url + remote, timeout=60).read()
    if sha256(content).hexdigest() != expected_hash:
        raise RuntimeError(f'Загруженный файл не совпадает с ожидаемой версией: {remote}')
    text = content.decode('utf-8')
    if local == 'edm_networks.py':
        text = text.replace('from torch_utils import persistence',
                            '# Persistence decorators omitted: this project saves state_dict checkpoints.')
        text = text.replace('@persistence.persistent_class', '')
    (vendor_dir / local).write_text(text, encoding='utf-8')
(vendor_dir / '__init__.py').touch()
(vendor_dir / 'SOURCE.md').write_text(base_url + 'training/networks.py', encoding='utf-8')
print('U-Net готова.')

## 3. Импорты и проверка устройства

In [ ]:
import csv
import gc
import json
import math
import random
import shutil
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import Compose, Resize, ToTensor
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage
from tqdm.auto import tqdm

from vendor.edm_networks import DhariwalUNet

version = tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2])
if version < (2, 6):
    raise RuntimeError('Нужен PyTorch >= 2.6. Используйте актуальную стандартную среду Colab.')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('Устройство:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU не обнаружен. Включите GPU в настройках Colab до начала обучения.')

## 4. Настройки

- `CONDITIONAL = True`: можно заказывать конкретную цифру.
- `EPOCHS`: число проходов по обучающему набору.
- `BATCH_SIZE`: сколько картинок обрабатывается за шаг обучения. При нехватке
  памяти снизьте с `128` до `64` или `32`, затем начните новый запуск.
- `LEARNING_RATE`: масштаб обновления параметров.
- `SEED`: начальная настройка генераторов случайных чисел.
- `USE_DRIVE = True`: подключить Google Drive и сохранять результат после каждой эпохи.
  Colab запросит доступ к Drive. По умолчанию подключение выключено.
- `TRAIN_MODEL = False`: пропустить обучение и загрузить готовый `.pth`.
  Тогда укажите `CHECKPOINT_PATH` или включите `UPLOAD_CHECKPOINT`.
- `ALLOW_CPU_TRAINING = True`: разрешить обучение без GPU, если это осознанно нужно.

In [ ]:
CONDITIONAL = True
EPOCHS = 6
BATCH_SIZE = 128
LEARNING_RATE = 3e-4
SEED = 42
PREVIEW_STEPS = 50

TRAIN_MODEL = True
ALLOW_CPU_TRAINING = False
USE_DRIVE = False
UPLOAD_CHECKPOINT = False
CHECKPOINT_PATH = ''

if EPOCHS < 1 or BATCH_SIZE < 1 or LEARNING_RATE <= 0 or PREVIEW_STEPS < 2:
    raise ValueError('epochs и batch size должны быть положительными, steps >= 2.')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    result_root = Path('/content/drive/MyDrive/MNIST_Diffusion')
else:
    result_root = Path('mnist_diffusion_runs')
run_name = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:6]
RUN_DIR = result_root / run_name
DATA_DIR = Path('mnist_data')
print('Папка результатов этого запуска:', RUN_DIR)

## 5. Данные: изображения цифр MNIST

MNIST содержит 60 000 обучающих и 10 000 тестовых примеров. Каждую картинку
уменьшаем до `16×16`.

In [ ]:
def make_loaders(data_dir, batch_size, workers, device):
    transform = Compose([Resize((16, 16)), ToTensor()])
    train = MNIST(data_dir, train=True, transform=transform, download=True)
    test = MNIST(data_dir, train=False, transform=transform, download=True)
    kwargs = dict(batch_size=batch_size, num_workers=workers, pin_memory=device.type == "cuda")
    return DataLoader(train, shuffle=True, **kwargs), DataLoader(test, shuffle=False, **kwargs)

@torch.no_grad()
def calc_data_std(loader):
    """Match the notebook: sqrt(mean over pixels of Var over images))."""
    total = torch.zeros(1, 16, 16, dtype=torch.float64)
    total2 = torch.zeros_like(total)
    count = 0
    for images, _ in loader:
        images = images.double()
        total += images.sum(0)
        total2 += images.square().sum(0)
        count += len(images)
    variance = (total2 / count - (total / count).square()).mean()
    return variance.clamp_min(0).sqrt().item()

In [ ]:
if TRAIN_MODEL:
    train_loader, test_loader = make_loaders(DATA_DIR, BATCH_SIZE, workers=0, device=DEVICE)
    example_images, example_labels = next(iter(test_loader))
    grid = make_grid(example_images[:40], nrow=10)
    plt.figure(figsize=(12, 4))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis('off')
    plt.title('Настоящие изображения из MNIST')
    plt.show()
    print('Форма пакета:', tuple(example_images.shape))
    print('Метки первых изображений:', example_labels[:10].tolist())
else:
    print('Обучение отключено: скачивание MNIST пропущено.')

## 6. Денойзер: U-Net с обёрткой EDM

U-Net содержит обучаемые параметры. Обёртка `EDMPrecond` масштабирует числа
и собирает **оценку чистого изображения**. Она получает:

```text
шумную картинку + уровень шума sigma + класс цифры (для условной модели)
```

Метка `10` означает отсутствие условия.
Денойзер возвращает отдельное изображение и не меняет входной тензор на месте.

In [ ]:
class EDMPrecond(nn.Module):
    """Turn the raw U-Net output into a clean-image estimate D(x, sigma)."""

    def __init__(self, sigma_data: float, conditional: bool = False):
        super().__init__()
        self.config = dict(sigma_data=float(sigma_data), conditional=conditional)
        self.sigma_data = float(sigma_data)
        self.conditional = conditional
        self.net = DhariwalUNet(
            img_resolution=16, in_channels=1, out_channels=1,
            model_channels=32, channel_mult=[1, 2, 3, 4],
            label_dim=11 if conditional else 0,
        )

    def forward(self, x, sigma, labels=None):
        # [B,1,1,1] makes one noise level broadcast over each image.
        sigma = torch.as_tensor(sigma, device=x.device, dtype=x.dtype).reshape(-1, 1, 1, 1)
        if sigma.numel() not in (1, x.shape[0]) or not torch.all(sigma > 0):
            raise ValueError("sigma must be positive, with one value or one per image")
        sigma = sigma.expand(x.shape[0], 1, 1, 1)
        if self.conditional:
            if labels is None:
                labels = torch.full((x.shape[0],), 10, device=x.device, dtype=torch.long)
            labels = F.one_hot(labels.long(), num_classes=11).to(dtype=x.dtype)
        elif labels is not None:
            raise ValueError("An unconditional model does not accept class labels")
        data2 = self.sigma_data ** 2
        c_skip = data2 / (data2 + sigma.square())
        c_out = self.sigma_data * sigma / (data2 + sigma.square()).sqrt()
        c_in = (data2 + sigma.square()).rsqrt()
        c_noise = sigma.log() / 4
        prediction = self.net(c_in * x, c_noise.flatten(), labels)
        return c_skip * x + c_out * prediction

## 7. Ошибка и обучение удалению шума

Мы сами добавляем шум к чистому изображению, поэтому знаем правильный ответ.

```python
noisy = clean + sigma * noise
prediction = model(noisy, sigma)
```

`edm_loss` сравнивает предсказание с оригиналом. При условном обучении примерно
10% меток заменяются на `10`, чтобы сеть умела работать и без указания цифры.
`evaluate` считает ошибку на проверочных данных, не обновляя параметры.

Loss — ошибка восстановления, а не процент правильных цифр. Тестовый MNIST здесь
используется для выбора лучшей эпохи, то есть играет роль validation.
Для независимой итоговой оценки нужен отдельный validation split.

In [ ]:
def edm_loss(model, clean, labels=None, mean=0.0, std=1.0,
             uncond_prob=0.1, generator=None):
    """Weighted denoising MSE. clean is in [-1,1]; labels are integer IDs."""
    batch = clean.shape[0]
    sigma = (mean + std * torch.randn(
        (batch, 1, 1, 1), device=clean.device, generator=generator
    )).exp()
    noise = torch.randn(clean.shape, device=clean.device, generator=generator) * sigma
    if model.conditional and labels is not None:
        # Do not modify the DataLoader's labels in-place.
        mask = torch.rand(batch, device=clean.device, generator=generator) < uncond_prob
        labels = torch.where(mask, 10, labels)
    denoised = model(clean + noise, sigma, labels)
    weight = (sigma.square() + model.sigma_data ** 2) / (
        sigma * model.sigma_data
    ).square()
    return (weight * (denoised - clean).square()).mean()

@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    # Fixed noise makes epoch-to-epoch test loss easier to compare.
    rng = torch.Generator(device=device).manual_seed(12345)
    total = 0.0
    count = 0
    for images, labels in loader:
        clean = images.to(device) * 2 - 1
        labels = labels.to(device) if model.conditional else None
        loss = edm_loss(model, clean, labels, uncond_prob=0.0, generator=rng)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite test loss")
        total += loss.item() * len(images)
        count += len(images)
    return total / count

## 8. Генерация: метод Эйлера

При генерации исходной картинки нет — начинаем со случайного шума.
На каждом шаге денойзер предлагает оценку чистой картинки, а метод Эйлера
вычисляет следующее состояние `x`, соответствующее меньшему уровню шума:

```python
denoised = model(x, sigma)
x_next = x + (x - denoised) / sigma * (next_sigma - sigma)
```

CFG усиливает влияние класса. В этой реализации `cfg=0` уже означает обычную
условную генерацию. Большие значения CFG не гарантируют лучшего качества.

In [ ]:
def get_timesteps(sigma_min=0.02, sigma_max=80.0, steps=50, rho=7.0, device="cpu"):
    if steps < 2 or not 0 < sigma_min < sigma_max or rho <= 0:
        raise ValueError("Need steps >= 2, 0 < sigma_min < sigma_max, rho > 0")
    ramp = torch.linspace(0, 1, steps, device=device)
    levels = (sigma_max ** (1 / rho) + ramp * (
        sigma_min ** (1 / rho) - sigma_max ** (1 / rho)
    )) ** rho
    return torch.cat((levels, levels.new_zeros(1)))

@torch.inference_mode()
def sample_euler(model, count=40, labels=None, steps=50, sigma_min=0.02,
                 sigma_max=80.0, rho=7.0, cfg=0.0, stochastic=False, seed=42):
    """Return raw images in model coordinates (approximately [-1,1]).

    cfg=0 means ordinary conditional sampling, as in the notebook.
    ODE: dx/dsigma = (x-D)/sigma; the sigma step is negative.
    SDE: double the drift and add sqrt(2*sigma*abs(delta_sigma))*N(0,I).
    """
    if count < 1 or cfg < 0:
        raise ValueError("count must be positive and cfg must be nonnegative")
    if cfg and (not model.conditional or labels is None):
        raise ValueError("CFG requires a conditional model and explicit digit labels")
    device = next(model.parameters()).device
    if labels is not None:
        labels = torch.as_tensor(labels, device=device, dtype=torch.long)
        if labels.shape != (count,) or not torch.all((labels >= 0) & (labels <= 10)):
            raise ValueError("Provide count labels in the range 0..10")
    model.eval()
    rng = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn((count, 1, 16, 16), device=device, generator=rng) * sigma_max
    levels = get_timesteps(sigma_min, sigma_max, steps, rho, device)
    for current, following in zip(levels[:-1], levels[1:]):
        denoised = model(x, current, labels)
        if cfg:
            uncond = model(x, current, torch.full_like(labels, 10))
            denoised = denoised + cfg * (denoised - uncond)
        drift = (x - denoised) / current
        delta = following - current
        x = x + (2 if stochastic else 1) * drift * delta
        if stochastic:
            x = x + (2 * current * delta.abs()).sqrt() * torch.randn(
                x.shape, device=device, generator=rng
            )
    return x

def save_samples(images, path, columns=10):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Clamp only for display; never feed clipped/normalized pictures to the solver.
    save_image((images.detach().cpu() * 0.5 + 0.5).clamp(0, 1), path, nrow=columns)

## 9. Сохранение и загрузка `.pth`

Checkpoint содержит веса и конфигурацию, необходимую для создания сети.
`last.pth` также хранит состояние Adam. `best.pth` — лучшие веса по проверочному loss.
Функции совместимы с checkpoint нашего локального Python-скрипта.

In [ ]:
def save_checkpoint(model, path, epoch=0, optimizer=None, metrics=None):
    """Only tensors and simple metadata: load with weights_only=True."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    state = {key: value.detach().cpu() for key, value in model.state_dict().items()}
    checkpoint = dict(format_version=1, model_config=model.config, model_state=state,
                      epoch=epoch, metrics=metrics or {})
    if optimizer is not None:
        checkpoint["optimizer_state"] = optimizer.state_dict()
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)

def load_checkpoint(path, device):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint.get("format_version") != 1:
        raise ValueError("Unsupported checkpoint format")
    model = EDMPrecond(**checkpoint["model_config"])
    model.load_state_dict(checkpoint["model_state"], strict=True)
    return model.to(device).eval()

## 10. Создать необученную модель

Диапазон пикселей для обучения переводим из `[0,1]` в `[-1,1]`:
`clean = images * 2 - 1`. Измеряем разброс данных и передаём его как `sigma_data`.
Это характеристика данных; `sigma` в denoising loss — сила добавленного шума.
Adam использует градиенты, чтобы менять параметры U-Net.

In [ ]:
if TRAIN_MODEL:
    if DEVICE.type != 'cuda' and not ALLOW_CPU_TRAINING:
        raise RuntimeError(
            'Обучение остановлено: включите GPU в настройках Colab и снова запустите ячейки. '
            'Для осознанного запуска на CPU задайте ALLOW_CPU_TRAINING=True.'
        )
    sigma_data = calc_data_std(train_loader) * 2
    model = EDMPrecond(sigma_data=sigma_data, conditional=CONDITIONAL).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    print('Параметров:', f'{sum(p.numel() for p in model.parameters()):,}')
    print('sigma_data:', sigma_data)
    print('Модель пока не обучена. Следующая ячейка начнёт обучение.')
else:
    model = None
    print('Создание случайной модели пропущено: будем загружать готовые веса.')

## 11. Обучение

Для каждого пакета:

1. Сбросить предыдущие градиенты: `zero_grad`.
2. Зашумить данные и вычислить ошибку: `edm_loss`.
3. Вычислить градиенты: `backward`.
4. Изменить параметры: `step`.

После каждой эпохи сохраняются `last.pth`, `best.pth`, `metrics.csv` и PNG.
На графике появится средняя ошибка по эпохам; первые изображения могут быть плохими.
При исчерпании памяти GPU уменьшите `BATCH_SIZE`, затем начните новый запуск с ячейки
настроек. Повторный запуск этой ячейки в той же папке не перезаписывает веса.

In [ ]:
history = []
if TRAIN_MODEL:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    if (RUN_DIR / 'last.pth').exists():
        raise FileExistsError('В этой папке уже есть обучение. Начните новый запуск с ячейки настроек.')
    config = dict(conditional=CONDITIONAL, epochs=EPOCHS, batch_size=BATCH_SIZE,
                  learning_rate=LEARNING_RATE, seed=SEED, device=str(DEVICE),
                  torch_version=str(torch.__version__), sigma_data=sigma_data)
    (RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    best_loss = math.inf
    with (RUN_DIR / 'metrics.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=['epoch', 'train_loss', 'test_loss'])
        writer.writeheader()
        for epoch in range(1, EPOCHS + 1):
            model.train()
            total, seen = 0.0, 0
            progress = tqdm(train_loader, desc=f'Эпоха {epoch}/{EPOCHS}')
            for images, labels in progress:
                clean = images.to(DEVICE) * 2 - 1
                labels = labels.to(DEVICE) if model.conditional else None
                optimizer.zero_grad(set_to_none=True)
                loss = edm_loss(model, clean, labels)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Loss стал NaN или бесконечностью.')
                loss.backward()
                optimizer.step()
                total += loss.item() * len(images)
                seen += len(images)
                progress.set_postfix(loss=f'{loss.item():.4f}')
            metrics = dict(epoch=epoch, train_loss=total / seen,
                           test_loss=evaluate(model, test_loader, DEVICE))
            history.append(metrics)
            writer.writerow(metrics)
            stream.flush()
            save_checkpoint(model, RUN_DIR / 'last.pth', epoch, optimizer, metrics)
            if metrics['test_loss'] < best_loss:
                best_loss = metrics['test_loss']
                save_checkpoint(model, RUN_DIR / 'best.pth', epoch, metrics=metrics)
            preview_labels = torch.arange(40, device=DEVICE) % 10 if model.conditional else None
            preview = sample_euler(model, count=40, labels=preview_labels,
                                   steps=PREVIEW_STEPS, seed=SEED)
            preview_path = RUN_DIR / f'epoch_{epoch:03d}.png'
            save_samples(preview, preview_path)
            print(f"Эпоха {epoch}: train={metrics['train_loss']:.5f}, "
                  f"test={metrics['test_loss']:.5f}")
            display(DisplayImage(filename=str(preview_path), width=800))
    plt.figure(figsize=(8, 4))
    plt.plot([row['epoch'] for row in history], [row['train_loss'] for row in history], label='train')
    plt.plot([row['epoch'] for row in history], [row['test_loss'] for row in history], label='test / validation')
    plt.xlabel('Эпоха')
    plt.ylabel('Denoising loss')
    plt.grid(True)
    plt.legend()
    plt.show()
    print('Обучение завершено. Веса:', RUN_DIR / 'best.pth')
else:
    print('Обучение отключено настройкой TRAIN_MODEL=False.')

## 12. Загрузить лучшую модель

После обучения автоматически загружается `best.pth`. Теперь используется лучшая
по loss эпоха, которая может отличаться от последней.

Если хотите использовать ранее сохранённые веса в новой сессии:
в настройках укажите `TRAIN_MODEL=False` и путь `CHECKPOINT_PATH` на Drive
либо `UPLOAD_CHECKPOINT=True` для загрузки с компьютера. Затем выполните ячейки
по порядку. Обучение и загрузка MNIST будут пропущены.

In [ ]:
if TRAIN_MODEL:
    selected_checkpoint = RUN_DIR / 'best.pth'
else:
    if UPLOAD_CHECKPOINT:
        from google.colab import files
        uploaded = files.upload()
        checkpoint_names = [name for name in uploaded if name.endswith(('.pth', '.pt'))]
        if len(checkpoint_names) != 1:
            raise ValueError('Загрузите ровно один checkpoint .pth или .pt.')
        selected_checkpoint = Path(checkpoint_names[0])
    elif CHECKPOINT_PATH:
        selected_checkpoint = Path(CHECKPOINT_PATH)
    else:
        raise ValueError('Укажите CHECKPOINT_PATH или включите UPLOAD_CHECKPOINT в настройках.')
    RUN_DIR.mkdir(parents=True, exist_ok=True)
if model is not None:
    del model
if 'optimizer' in globals():
    del optimizer
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
model = load_checkpoint(selected_checkpoint, DEVICE)
print('Загружено:', selected_checkpoint)
print('Условная модель:', model.conditional)

## 13. Сгенерировать выбранную цифру

Изменяйте `DIGIT`, `CFG`, `GENERATION_SEED` и повторно запускайте только эту ячейку.
Повторное обучение не требуется. `DIGIT=None` создаёт классы 0–9 по порядку.
При безусловной модели метки не используются и CFG автоматически равен нулю.

`STOCHASTIC=False` — детерминированный процесс; `True` — добавление шума на шагах.
`GENERATION_STEPS` — число обновлений Эйлера. `COUNT` — число изображений.

In [ ]:
DIGIT = 7
COUNT = 40
CFG = 2.0
GENERATION_STEPS = 50
GENERATION_SEED = 123
STOCHASTIC = False

if DIGIT is not None and DIGIT not in range(10):
    raise ValueError('DIGIT должен быть от 0 до 9 или None.')
if model.conditional:
    labels = (torch.arange(COUNT, device=DEVICE) % 10 if DIGIT is None else
              torch.full((COUNT,), DIGIT, device=DEVICE, dtype=torch.long))
    actual_cfg = CFG
else:
    labels, actual_cfg = None, 0.0
    print('Безусловная модель: выбор цифры и CFG не применяются.')
generated = sample_euler(model, count=COUNT, labels=labels, steps=GENERATION_STEPS,
                         cfg=actual_cfg, stochastic=STOCHASTIC, seed=GENERATION_SEED)
if not torch.isfinite(generated).all():
    raise FloatingPointError('При генерации получились NaN или бесконечности.')
generated_path = RUN_DIR / 'generated.png'
save_samples(generated, generated_path)
generation_settings = dict(digit=DIGIT if model.conditional else None,
                           labels=labels.cpu().tolist() if labels is not None else None,
                           cfg=actual_cfg, steps=GENERATION_STEPS, seed=GENERATION_SEED,
                           count=COUNT, stochastic=STOCHASTIC,
                           checkpoint=str(selected_checkpoint))
(RUN_DIR / 'generated.json').write_text(json.dumps(generation_settings, indent=2), encoding='utf-8')
display(DisplayImage(filename=str(generated_path), width=800))
print('PNG сохранён:', generated_path)

## 14. Посмотреть именно работу денойзера

Здесь берём настоящее изображение, сами добавляем шум и **один раз** вызываем
денойзер. Метод Эйлера в этой ячейке не используется.

Строки картинки: **оригинал → шумное изображение → предсказание денойзера**.
Изменяйте `DEMO_SIGMA`, чтобы увидеть влияние уровня шума. При очень сильном шуме
сеть не обязана восстановить исходное изображение: информации может не хватать.
Изображения при показе обрезаются до `[0,1]`, но вход денойзера не обрезается.

In [ ]:
DEMO_SIGMA = 0.5
if DEMO_SIGMA <= 0:
    raise ValueError('DEMO_SIGMA должен быть положительным.')
if TRAIN_MODEL:
    demo_images, demo_labels = next(iter(test_loader))
    clean = demo_images[:8].to(DEVICE) * 2 - 1
    demo_labels = demo_labels[:8].to(DEVICE) if model.conditional else None
    demo_rng = torch.Generator(device=DEVICE).manual_seed(123)
    noisy = clean + DEMO_SIGMA * torch.randn(clean.shape, device=DEVICE, generator=demo_rng)
    model.eval()
    with torch.inference_mode():
        denoised = model(noisy, DEMO_SIGMA, demo_labels)
    comparison = torch.cat([clean, noisy, denoised])
    comparison_path = RUN_DIR / 'denoising.png'
    save_samples(comparison, comparison_path, columns=8)
    display(DisplayImage(filename=str(comparison_path), width=800))
    print('Сверху вниз: оригинал, шум, оценка денойзера.')
else:
    print('Демонстрация пропущена: в режиме готовых весов MNIST не загружался.')

## 15. Скачать результаты

Запускать необязательно. Установите `DOWNLOAD_RESULTS=True` и выполните ячейку,
чтобы скачать ZIP с весами, метриками, настройками, PNG и исходником сети.
Архив также включает код функций модели, чтобы checkpoint можно было восстановить.
При использовании Google Drive файлы уже сохранены в выбранной папке Drive.
Сам ноутбук сохраняйте отдельно через меню Colab.

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    # Собираем определения из выполненных ячеек, включая пользовательские правки.
    import ast
    names = ['EDMPrecond', 'edm_loss', 'get_timesteps', 'sample_euler',
             'save_samples', 'save_checkpoint', 'load_checkpoint']
    definitions = {}
    for source in get_ipython().history_manager.input_hist_raw:
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, (ast.ClassDef, ast.FunctionDef)) and node.name in names:
                definitions[node.name] = ast.unparse(node)
    if set(names) - definitions.keys():
        raise RuntimeError('Сначала выполните ячейки с определениями модели и функций.')
    imports = ('from pathlib import Path\nimport torch\nfrom torch import nn\n'
               'from torch.nn import functional as F\n'
               'from torchvision.utils import save_image\n'
               'from vendor.edm_networks import DhariwalUNet\n\n')
    export_source = imports + '\n\n'.join(definitions[name] for name in names)
    (RUN_DIR / 'model_definition.py').write_text(export_source, encoding='utf-8')
    shutil.copytree('vendor', RUN_DIR / 'vendor', dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('__pycache__'))
    if not (RUN_DIR / 'best.pth').exists():
        shutil.copy2(selected_checkpoint, RUN_DIR / 'best.pth')
    (RUN_DIR / 'RESTORE.txt').write_text(
        'Install torch>=2.6, torchvision>=0.21, numpy and Pillow.\n'
        'From this folder run:\n'
        'from model_definition import load_checkpoint\n'
        'model = load_checkpoint("best.pth", "cpu")\n', encoding='utf-8')
    archive_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
    files.download(archive_path)
else:
    print('Для скачивания архива установите DOWNLOAD_RESULTS=True и повторите ячейку.')